In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Jeffery's orbits of a rigid body

### Imports

In [ ]:
import jax.numpy as jnp
import numpy as np

import softmobility as sm
from softmobility.tutorials import figstyle

figstyle.apply()
np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Body shape 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:     # Design parameter prefixes/names (extends default "design")
  - myradius      # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)
  - distance
  - offset

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `design_names`
  myradius1: 1.
  distance: 1.

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0
    position: [offset - myradius0 - distance/2, 0, 0]

  - # Sphere 1 #################
    radius: myradius1
    position: [offset + myradius1 + distance/2, 0, 0]
"""

In [3]:
mybody= sm.SoftBody(yaml_data, verbose=False)
print(repr(mybody))

SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  4 design parameters
  0 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: ['distance', 'myradius0', 'myradius1', 'offset'] = [ 1.  1.  1.  0.]
  input parameters param: []

SPHERE 0
  radius: 1.0
  position: [-1.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  c_field:
[]
  c_stiff:
[]

SPHERE 1
  radius: 1.0
  position: [ 1.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  c_field:
[]
  c_stiff:
[]



## Fluid-structure interaction problem

In [4]:
# Create a shear flow with shear rate 1
shear_flow = sm.shear_flow(shear_rate=1.0)

# Test it
pos = [1.0, 2.0, 3.0]
grad_u = shear_flow.gradient(pos)
Omega, rate_of_strain = shear_flow.omega_rate_of_strain(pos)
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain E):\n", rate_of_strain)

Velocity Gradient Tensor (∇u):
 [[ 0.  1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Angular velocity Ω: [ 0.   0.  -0.5]
Rate-of-strain E):
 [[ 0.   0.5  0. ]
 [ 0.5  0.   0. ]
 [ 0.   0.   0. ]]


### Numerics

In [5]:
# parameters
final_time = 30
dt         = 0.02
n_steps    = int(final_time / dt)

# simulation
rollout = sm.FlowBodyRollout(soft_body=mybody, flow=shear_flow)
positions, orientations, dofs = rollout.rollout(dt=dt, n_steps=n_steps, init_orientation=jnp.array([0., 0.6, 0.]))

### Figure setup

In [ ]:
# Time vector — starts at dt because the rollout output is post-step.
t = jnp.linspace(dt, final_time, n_steps)

# Two-row figure: position components on top, orientation components below.
fig, axes = figstyle.subplots(size="full", aspect=1.0, nrows=2, sharex=True)

axes[0].plot(t, positions[:, 0], color=figstyle.COLORS["red"], label="Position X")
axes[0].plot(t, positions[:, 1], color=figstyle.COLORS["blue"], label="Position Y")
axes[0].plot(t, positions[:, 2], color=figstyle.COLORS["black"], label="Position Z")
axes[0].set_ylabel("Position")
axes[0].set_title("Trajectory over time")
axes[0].legend(frameon=False, loc="upper right")

axes[1].plot(t, orientations[:, 0], color=figstyle.COLORS["red"], label="Orientation X")
axes[1].plot(t, orientations[:, 1], color=figstyle.COLORS["blue"], label="Orientation Y")
axes[1].plot(t, orientations[:, 2], color=figstyle.COLORS["black"], label="Orientation Z")
axes[1].set_xlabel("t")
axes[1].set_ylabel("Orientation")
axes[1].set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
                   ["$-\\pi$", "$-\\pi/2$", "0", "$\\pi/2$", "$\\pi$"])
axes[1].legend(frameon=False, loc="upper right")

### Figure

In [ ]:
fig

### Theory

In [ ]:
phi_num = np.zeros_like(t)
theta_num = np.zeros_like(t)
p0 = np.array([1, 0, 0])

for i, _ti in enumerate(t):
    R = np.asarray(sm.rotation_matrix(orientations[i]))
    p = R @ p0
    phi_num[i] = np.atan2(p[1], p[0])
    theta_num[i] = np.atan(np.sqrt((1 - p[1]**2 - p[0]**2) / (p[1]**2 + p[0]**2)))

In [10]:
tensors = mybody.compute_tensors()
M0 = tensors.M @ mybody.compute_composition_of_forces()  # Mobility with forces/torques at the ref point x_0
N = mybody.Nspheres
# Reshape M into a 3D array where each element is a 6x6 matrix representing one sphere's data
M_blocks = M0.reshape(6, 6 * N).T.reshape(N, 6, 6)
M_mean = np.mean(M_blocks, axis=0)

# Bretherton parameter
Bretherton = tensors.C_E[-1,1] # * M_mean[1,1]

# print("Rigid mobility tensor\n", M_mean)
# print("Coupling with strain\n", tensors.C_E)    # Exx, Exy, Exz, Eyy, Eyz
print("Bretherton parameters:", Bretherton)

Bretherton parameters: 0.7168667


In [11]:
time = np.linspace(0,t[-1],num=100)
phi0 = phi_num[0]
theta0 = theta_num[0]
c = np.sqrt((1 + Bretherton) / (1 - Bretherton))
K = np.sqrt(np.cos(phi0)**2 + c**2 * np.sin(phi0)) / np.tan(theta0)

phi = np.atan2(-(1/c) * np.sin(time / (c + 1/c)), np.cos(time / (c + 1/c)))
# phi = np.atan(-(1/c) * np.tan(time / (c + 1/c)))
theta = np.atan(np.sqrt(np.cos(phi)**2 + c**2 * np.sin(phi)**2) / K)

### Figure setup

In [ ]:
fig, ax = figstyle.figure(size="full", aspect=1.4)
ax.plot(t, phi_num, color=figstyle.COLORS["red"], label=r"Numerics $\varphi$")
ax.plot(t, theta_num, color=figstyle.COLORS["blue"], label=r"Numerics $\theta$")
ax.plot(time, phi, marker="o", linestyle="none",
        color=figstyle.COLORS["red"], markerfacecolor="white", markersize=4,
        label=r"Theory $\varphi$")
ax.plot(time, theta, marker="o", linestyle="none",
        color=figstyle.COLORS["blue"], markerfacecolor="white", markersize=4,
        label=r"Theory $\theta$")
ax.set_xlabel("t")
ax.set_ylabel(r"$\varphi,\, \theta$")
ax.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
              ["$-\\pi$", "$-\\pi/2$", "0", "$\\pi/2$", "$\\pi$"])
ax.set_title("Comparison of Jeffery's orbits")
ax.legend(frameon=False, loc="upper right")

### Figure

In [ ]:
fig